# AqSolDB Data Cleaning and Morgan Fingerprint Preparation

Place the original **curated-solubility-dataset.csv** in the notebook working directory and run all cells in order. This notebook checks the source data, generates Morgan fingerprints from SMILES, and exports records with valid fingerprints.

The output retains InChI and InChIKey alongside the fingerprint columns so downstream analyses can match records or group molecules by structure. It also retains the measured solubility and existing descriptors.

[AqSolDB data source (Sorkun et al.)](https://pmc.ncbi.nlm.nih.gov/articles/PMC6687799/)

## 1. Load the Data

Read the original AqSolDB CSV and inspect the first few records.

In [1]:
# 导入 Path，用于表示和检查文件路径
# Import Path to represent and check file paths
from pathlib import Path

# 导入 pandas，用于读取和处理表格；后面用 pd 作为简称
# Import pandas for reading and processing tables, using the alias pd
import pandas as pd

# 导入 NumPy，用于后续数值检查；后面用 np 作为简称
# Import NumPy for later numeric checks, using the alias np
import numpy as np

# 导入 display，用于在 notebook 中清楚地显示表格
# Import display to show tables in the notebook
from IPython.display import display

# 指定当前工作目录中的 CSV 文件，不使用个人电脑的绝对路径
# Specify the CSV in the working directory using a relative path
DATA_PATH = Path("curated-solubility-dataset.csv")

# 如果指定位置没有这个文件，就执行下面缩进的语句
# Run the indented statement if the specified file does not exist
if not DATA_PATH.is_file():

    # 停止运行，并提示上传文件，避免继续处理不存在的数据
    # Stop execution and ask for the file to be uploaded
    raise FileNotFoundError("Place the original CSV in the notebook working directory.")

# 读取 CSV，并把读入的原始数据表命名为 raw
# Read the CSV and store the original table as raw
raw = pd.read_csv(DATA_PATH)

# 复制 raw，得到后续用于清洗的数据表 df
# Copy raw to create df, the working table used for cleaning
df = raw.copy()

# 显示文件名；name 表示路径中的文件名称
# Show the filename; name returns the filename part of the path
print(f"Loaded file: {DATA_PATH.name}")

# len(raw) 计算行数；:, 让数字按千位分隔，例如 9,982
# len(raw) counts rows; :, adds thousands separators, such as 9,982
print(f"Rows: {len(raw):,}")

# shape 是（行数，列数）；[1] 取其中第二个数，也就是列数
# shape contains (rows, columns); [1] selects the column count
print(f"Columns: {raw.shape[1]}")

# head(5) 取前五行，display 将它们显示出来，完整数据仍然保留
# head(5) selects the first five rows; display shows them without removing other rows
display(raw.head(5))

Loaded file: curated-solubility-dataset.csv
Rows: 9,982
Columns: 26


,ID,Name,InChI,InChIKey,SMILES,Solubility,SD,Ocurrences,Group,MolWt,MolLogP,MolMR,HeavyAtomCount,NumHAcceptors,NumHDonors,NumHeteroatoms,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumSaturatedRings,NumAliphaticRings,RingCount,TPSA,LabuteASA,BalabanJ,BertzCT
0,A-3,"N,N,N-trimethyloctadecan-1-aminium bromide",InChI=1S/C21H46N.BrH/c1-5-6-7-8-9-10-11-12-13-...,SZEMGTQCPRNXEG-UHFFFAOYSA-M,[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C,-3.616127,0.0,1,G1,392.510,3.9581,102.4454,23.0,0.0,0.0,2.0,17.0,142.0,0.0,0.0,0.0,0.0,0.00,158.520601,0.000000e+00,210.377334
1,A-4,Benzo[cd]indol-2(1H)-one,InChI=1S/C11H7NO/c13-11-8-5-1-3-7-4-2-6-9(12-1...,GPYLCFQEKPUWLD-UHFFFAOYSA-N,O=C1Nc2cccc3cccc1c23,-3.254767,0.0,1,G1,169.183,2.4055,51.9012,13.0,1.0,1.0,2.0,0.0,62.0,2.0,0.0,1.0,3.0,29.10,75.183563,2.582996e+00,511.229248
2,A-5,4-chlorobenzaldehyde,InChI=1S/C7H5ClO/c8-7-3-1-6(5-9)2-4-7/h1-5H,AVPYQKSLYISFPO-UHFFFAOYSA-N,Clc1ccc(C=O)cc1,-2.177078,0.0,1,G1,140.569,2.1525,36.8395,9.0,1.0,0.0,2.0,1.0,46.0,1.0,0.0,0.0,1.0,17.07,58.261134,3.009782e+00,202.661065
3,A-8,"zinc bis[2-hydroxy-3,5-bis(1-phenylethyl)benzo...",InChI=1S/2C23H22O3.Zn/c2*1-15(17-9-5-3-6-10-17...,XTUPUYCJWKHGSW-UHFFFAOYSA-L,[Zn++].CC(c1ccccc1)c2cc(C(C)c3ccccc3)c(O)c(c2)...,-3.924409,0.0,1,G1,756.226,8.1161,200.7106,53.0,6.0,2.0,7.0,10.0,264.0,6.0,0.0,0.0,6.0,120.72,323.755434,2.322963e-07,1964.648666
4,A-9,4-({4-[bis(oxiran-2-ylmethyl)amino]phenyl}meth...,InChI=1S/C25H30N2O4/c1-5-20(26(10-22-14-28-22)...,FAUAZXVRLVIARB-UHFFFAOYSA-N,C1OC1CN(CC2CO2)c3ccc(Cc4ccc(cc4)N(CC5CO5)CC6CO...,-4.662065,0.0,1,G1,422.525,2.4854,119.0760,31.0,6.0,0.0,6.0,12.0,164.0,2.0,4.0,4.0,6.0,56.60,183.183268,1.084427e+00,769.899934


## 2. Define the Target and Check the Descriptors

Identify measured solubility and the 17 existing numerical descriptors to check their data types and values.

In [2]:
# 指定要预测的列；TARGET 是我们自己取的变量名
# Specify the column to predict; TARGET is a variable name we chose
TARGET = "Solubility"

# 用列表保存 17 个输入属性的列名
# Store the names of the 17 input features in a list
FEATURES = [
    "MolWt",                 # 分子量 / Molecular weight
    "MolLogP",               # 计算得到的亲油程度指标 / Calculated lipophilicity
    "MolMR",                 # 摩尔折射度，与分子大小和极化能力有关 / Molar refractivity
    "HeavyAtomCount",        # 非氢原子数量 / Number of non-hydrogen atoms
    "NumHAcceptors",         # 氢键受体数量 / Number of hydrogen-bond acceptors
    "NumHDonors",            # 氢键供体数量 / Number of hydrogen-bond donors
    "NumHeteroatoms",        # 非碳、非氢原子数量 / Number of heteroatoms
    "NumRotatableBonds",     # 可旋转键数量 / Number of rotatable bonds
    "NumValenceElectrons",   # 价电子数量 / Number of valence electrons
    "NumAromaticRings",      # 芳香环数量 / Number of aromatic rings
    "NumSaturatedRings",     # 饱和环数量 / Number of saturated rings
    "NumAliphaticRings",     # 非芳香环数量 / Number of non-aromatic rings
    "RingCount",             # 环的总数量 / Total number of rings
    "TPSA",                  # 拓扑极性表面积 / Topological polar surface area
    "LabuteASA",             # 估算的分子表面积 / Approximate molecular surface area
    "BalabanJ",              # 描述分子连接结构的指标 / Molecular topology index
    "BertzCT",               # 描述分子结构复杂程度的指标 / Molecular complexity index
]                            # 输入属性列表结束 / End of the feature list

# 将输入属性列名和目标列名合并，方便一起查看
# Combine feature names and the target name for inspection
model_columns = FEATURES + [TARGET]

# len() 计算列表中有多少个列名
# len() counts the number of names in the list
print(f"Number of input features: {len(FEATURES)}")

# 显示我们指定的预测目标列名
# Show the selected prediction target
print(f"Prediction target: {TARGET}")

# 选择这些列，并展示前五行
# Select these columns and display the first five rows
display(df[model_columns].head(5))

Number of input features: 17
Prediction target: Solubility


,MolWt,MolLogP,MolMR,HeavyAtomCount,NumHAcceptors,NumHDonors,NumHeteroatoms,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumSaturatedRings,NumAliphaticRings,RingCount,TPSA,LabuteASA,BalabanJ,BertzCT,Solubility
0,392.510,3.9581,102.4454,23.0,0.0,0.0,2.0,17.0,142.0,0.0,0.0,0.0,0.0,0.00,158.520601,0.000000e+00,210.377334,-3.616127
1,169.183,2.4055,51.9012,13.0,1.0,1.0,2.0,0.0,62.0,2.0,0.0,1.0,3.0,29.10,75.183563,2.582996e+00,511.229248,-3.254767
2,140.569,2.1525,36.8395,9.0,1.0,0.0,2.0,1.0,46.0,1.0,0.0,0.0,1.0,17.07,58.261134,3.009782e+00,202.661065,-2.177078
3,756.226,8.1161,200.7106,53.0,6.0,2.0,7.0,10.0,264.0,6.0,0.0,0.0,6.0,120.72,323.755434,2.322963e-07,1964.648666,-3.924409
4,422.525,2.4854,119.0760,31.0,6.0,0.0,6.0,12.0,164.0,2.0,4.0,4.0,6.0,56.60,183.183268,1.084427e+00,769.899934,-4.662065


## 3. Trim Text Fields

Remove leading and trailing whitespace from molecule names and structure fields.

In [3]:
# 从原始数据重新建立工作副本，方便重复运行这一格
# Create a fresh working copy so this cell can be rerun consistently
df = raw.copy()

# 列出需要检查的文本列
# List the text columns to inspect
text_columns = ["ID", "Name", "InChI", "InChIKey", "SMILES", "Group"]

# 建立一个记录表，用于保存“列名：修改数量”
# Create a dictionary to store each column name and its change count
trim_counts = {}

# 依次处理列表中的每一列；缩进的代码会对每列重复执行
# Process each listed column by repeating the indented instructions
for column in text_columns:

    # 取出当前列，并明确按文本处理
    # Select the current column and treat its values as text
    before = df[column].astype("string")

    # 去除每个值开头和结尾的空格、换行等空白
    # Remove spaces, line breaks, and other whitespace from both ends
    after = before.str.strip()

    # 比较处理前后；True 表示有变化，缺失的比较结果按 False 计
    # Compare before and after; True means changed, and missing comparisons count as False
    changed = before.ne(after).fillna(False)

    # 将 True 的数量相加，记录当前列修改了多少个值
    # Count the True values and record how many entries changed
    trim_counts[column] = int(changed.sum())

    # 保存处理结果；如果只剩空字符串，则标记为缺失值
    # Save the result and mark empty strings as missing values
    df[column] = after.replace("", pd.NA)

# 把各列的修改数量整理成一列，命名为 Trimmed values
# Organize the change counts into a series named Trimmed values
whitespace_report = pd.Series(trim_counts, name="Trimmed values")

# 转成表格并显示，方便查看每一列的处理结果
# Convert the series to a table and display the results
display(whitespace_report.to_frame())

,Trimmed values
ID,0
Name,55
InChI,0
InChIKey,0
SMILES,40
Group,0


## 4. Check Missing Values and Numeric Formats

Verify that solubility and the descriptors contain valid numeric values and summarize missing entries.

In [4]:
# 在模型相关列的基础上，加上两项数值型测量记录
# Add two numeric measurement fields to the model-related columns
numeric_columns = model_columns + ["SD", "Ocurrences"]

# 记录转换前各列的数据类型，方便比较
# Record each column's original data type for comparison
types_before = df[numeric_columns].dtypes.astype(str)

# 依次检查这些数值列
# Check each numeric column in turn
for column in numeric_columns:

    # 将当前列转换为数字；遇到无法转换的内容时直接报告错误
    # Convert the column to numbers and raise an error for invalid entries
    df[column] = pd.to_numeric(df[column], errors="raise")

# 建立检查表，每一行对应原数据中的一个数值列
# Create a report with one row for each numeric column
quality_report = pd.DataFrame(index=numeric_columns)

# 显示转换前的数据类型
# Show the data types before conversion
quality_report["Type before"] = types_before

# 显示转换后的数据类型
# Show the data types after conversion
quality_report["Type after"] = df[numeric_columns].dtypes.astype(str)

# isna() 找出缺失位置，sum() 统计每列的缺失数量
# isna() identifies missing entries; sum() counts them in each column
quality_report["Missing values"] = df[numeric_columns].isna().sum()

# isinf() 找出正、负无穷值，sum() 统计每列的数量
# isinf() identifies positive and negative infinity; sum() counts them
quality_report["Infinite values"] = np.isinf(df[numeric_columns]).sum()

# 显示数值检查表
# Display the numeric quality report
display(quality_report)

# 统计整张表中所有列的缺失值，包括文本列
# Count missing values across the entire table, including text columns
total_missing = int(df.isna().sum().sum())

# 显示整张表的缺失值总数
# Show the total number of missing values in the table
print(f"Total missing values: {total_missing}")

Total missing values: 0


,Type before,Type after,Missing values,Infinite values
MolWt,float64,float64,0,0
MolLogP,float64,float64,0,0
MolMR,float64,float64,0,0
HeavyAtomCount,float64,float64,0,0
NumHAcceptors,float64,float64,0,0
NumHDonors,float64,float64,0,0
NumHeteroatoms,float64,float64,0,0
NumRotatableBonds,float64,float64,0,0
NumValenceElectrons,float64,float64,0,0
NumAromaticRings,float64,float64,0,0


## 5. Check Duplicate Records

Remove exact duplicate rows and check the uniqueness of molecule identifiers.

In [5]:
# 记录处理前的行数
# Record the number of rows before duplicate removal
rows_before = len(df)

# 检查整行完全相同的记录；第一条之后的相同行记为重复
# Identify exact duplicate rows, counting copies after the first occurrence
exact_duplicates = int(df.duplicated(keep="first").sum())

# 对完全相同的记录保留第一条，并保存处理后的工作副本
# Keep the first occurrence of each exact record and save the working copy
df = df.drop_duplicates(keep="first").copy()

# 记录处理后的行数
# Record the number of rows after duplicate removal
rows_after = len(df)

# 显示处理前的行数
# Show the number of rows before removal
print(f"Rows before removal: {rows_before:,}")

# 显示删除的完全重复行数
# Show how many exact duplicate rows were removed
print(f"Exact duplicate rows removed: {exact_duplicates}")

# 显示处理后的行数
# Show the number of rows after removal
print(f"Rows after removal: {rows_after:,}")

# 列出需要单独检查的编号与分子结构标识列
# List the identifier and molecular representation columns to inspect
id_columns = ["ID", "InChI", "InChIKey", "SMILES"]

# 建立记录，用于保存每一列的重复数量
# Create a dictionary to store duplicate counts for each column
duplicate_counts = {}

# 依次检查这些列
# Check each listed column in turn
for column in id_columns:

    # 只根据当前列统计重复值，第一条之后的相同值记为重复
    # Count repeated values in this column after their first occurrence
    duplicate_counts[column] = int(df[column].duplicated(keep="first").sum())

# 将各列的重复数量整理成一列
# Organize the duplicate counts into a series
identity_report = pd.Series(duplicate_counts, name="Repeated entries")

# 转成表格并显示
# Convert the series to a table and display it
display(identity_report.to_frame())

Rows before removal: 9,982
Exact duplicate rows removed: 0
Rows after removal: 9,982


,Repeated entries
ID,0
InChI,0
InChIKey,0
SMILES,0


## 6. Review Numeric Ranges

Summarize numerical ranges and flag negative BalabanJ values for recalculation.

In [6]:
# 计算每个数值列的最小值和最大值，并将变量名称排成行
# Calculate each numeric column's minimum and maximum, with variables shown as rows
range_report = df[numeric_columns].agg(["min", "max"]).T

# 显示数值范围表
# Display the numeric range report
display(range_report)

# 找出 BalabanJ 小于 0 的位置；符合条件的行会标记为 True
# Identify rows where BalabanJ is below zero; matching rows are marked True
negative_balaban = df["BalabanJ"] < 0

# 指定核查时需要查看的列，包括分子身份、指标值和溶解度
# Specify columns needed for review, including molecular identity, the descriptor, and solubility
review_columns = ["ID", "Name", "SMILES", "BalabanJ", TARGET]

# 复制符合条件的记录，建立单独的待核查表
# Copy matching records into a separate review table
review_rows = df.loc[negative_balaban, review_columns].copy()

# 显示待核查记录的数量
# Show the number of records requiring review
print(f"Records with negative BalabanJ: {len(review_rows)}")

# 预览待核查表的前五行，完整结果仍保存在 review_rows 中
# Preview the first five records; all matching records remain in review_rows
display(review_rows.head(5))

Records with negative BalabanJ: 264


,min,max
MolWt,9.012000,5299.456000
MolLogP,-40.873200,68.541140
MolMR,0.000000,1419.351700
HeavyAtomCount,1.000000,388.000000
NumHAcceptors,0.000000,86.000000
NumHDonors,0.000000,26.000000
NumHeteroatoms,0.000000,89.000000
NumRotatableBonds,0.000000,141.000000
NumValenceElectrons,0.000000,2012.000000
NumAromaticRings,0.000000,35.000000


,ID,Name,SMILES,BalabanJ,Solubility
17,A-25,zinc bis[O-(2-ethylhexyl) O-isobutyl dithiopho...,[Zn++].CCCCC(CC)CO[P]([S-])(=S)OCC(C)C.CCCCC(C...,-5.688889e-07,-2.958195
22,A-30,aluminium(3+) ion sodium bis(oxosilanebis(olate)),[Na+].[Al+3].[O-][Si]([O-])=O.[O-][Si]([O-])=O,-3.000000e-08,-3.473139
25,A-33,"Phosphorodithioic acid, mixed O,O-bis(sec-Bu a...",[Zn++].CCC(C)O[P]([S-])(=S)OC(C)CC(C)C.CCC(C)O...,-4.900000e-07,-2.990877
53,A-76,pentacalcium hydroxide triphosphate,[OH-].[Ca++].[Ca++].[Ca++].[Ca++].[Ca++].[O-][...,-1.285714e-08,-4.141901
110,A-184,{2-hydroxy-3-[(9Z)-octadec-9-enoyloxy]propoxy}...,CCCCCCCC\C=C/CCCCCCCC(=O)OCC(O)COB(O)O.CCCCCCC...,-2.507398e-06,-5.345705


## 7. Read Molecular Structures with RDKit

Read each SMILES as a molecular structure. Records with unreadable SMILES go into the review file. Readable structures proceed to fingerprint generation.

In [7]:
# 如果环境里还没有 RDKit，就安装前次实验使用的版本
# Install the RDKit release used in the earlier experiment when needed
from importlib.util import find_spec

if find_spec("rdkit") is None:
    %pip install -q rdkit==2026.9.1

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator

print(f"RDKit version {rdBase.rdkitVersion}")

RDKit version 2026.09.1


In [8]:
# 每行读取一次 SMILES，成功得到的分子稍后也用于生成指纹
# Read each SMILES once and reuse the parsed molecules for fingerprint generation
with rdBase.BlockLogs():
    molecules = df["SMILES"].map(
        lambda text: Chem.MolFromSmiles(text) if pd.notna(text) else None
    )

# RDKit 返回有原子的分子时，此行通过结构检查
# A row passes when RDKit returns a molecule containing at least one atom
valid_smiles = molecules.map(lambda mol: mol is not None and mol.GetNumAtoms() > 0)
retained = df.loc[valid_smiles].copy()
rejected = df.loc[~valid_smiles].copy()

# 保留失败记录的所有结构和数值字段，并写明原因
# Retain the data fields for rejected records and document the reason
rejected["Review reason"] = "RDKit could not read a valid molecule from SMILES"
rejected.loc[rejected["SMILES"].isna(), "Review reason"] = "Missing SMILES"

print(f"Readable SMILES: {len(retained):,}")
print(f"Unreadable SMILES: {len(rejected):,}")
display(rejected[["ID", "Name", "SMILES", "Review reason"]])

Readable SMILES: 9,980
Unreadable SMILES: 2


,ID,Name,SMILES,Review reason
4792,B-1579,3-methyl-n-oxidepyridine,CC1=CC=C[NH++]([O-])[CH-]1,RDKit could not read a valid molecule from SMILES
5043,B-1866,n-oxidenicotinic acid,OC(=O)C1=C[NH++]([O-])[CH-]C=C1,RDKit could not read a valid molecule from SMILES


## 8. Recheck Negative BalabanJ Values

Recalculate BalabanJ for readable structures flagged in section 6, then show the agreement count and any differences.

In [9]:
# 选出待核查记录中可以读取结构的行
# Select rows from the review list that have readable structures
reviewed_molecules = molecules.loc[valid_smiles & (df["BalabanJ"] < 0)]
comparison = review_rows.loc[reviewed_molecules.index, ["ID", "BalabanJ"]].copy()

# 计算数值、比较原值，并记录结构包含的片段数
# Recalculate the descriptor, compare values, and count molecular fragments
comparison["Recalculated"] = reviewed_molecules.map(Descriptors.BalabanJ)
comparison["Matches"] = np.isclose(
    comparison["BalabanJ"], comparison["Recalculated"], rtol=1e-6, atol=1e-12
)
comparison["Parts"] = reviewed_molecules.map(lambda mol: len(Chem.GetMolFrags(mol)))

display(comparison.head())
print(f"Recalculations matching the recorded value: {comparison['Matches'].sum():,} of {len(comparison):,}")
print(f"Flagged records with unreadable SMILES: {len(review_rows) - len(comparison):,}")
if (~comparison["Matches"]).any():
    display(comparison.loc[~comparison["Matches"]])

Recalculations matching the recorded value: 264 of 264
Flagged records with unreadable SMILES: 0


,ID,BalabanJ,Recalculated,Matches,Parts
17,A-25,-5.688889e-07,-5.688889e-07,True,3
22,A-30,-3.000000e-08,-3.000000e-08,True,4
25,A-33,-4.900000e-07,-4.900000e-07,True,3
53,A-76,-1.285714e-08,-1.285714e-08,True,9
110,A-184,-2.507398e-06,-2.507398e-06,True,3


## 9. Generate and Validate Morgan Fingerprints

Use radius **2**, **1,024 bits**, and **includeChirality=False**. Generate each fingerprint individually and check that it contains exactly 1,024 values, each either 0 or 1. Exclude any record that fails fingerprint generation or validation and save its ID and reason for review.

[RDKit Morgan fingerprint documentation](https://www.rdkit.org/docs/GettingStartedInPython.html#morgan-fingerprints-circular-fingerprints)

In [10]:
fingerprint_size = 1024
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=fingerprint_size,
    includeChirality=False,
)

# 逐条生成指纹。某条生成失败时记录原因，然后继续处理下一条
# Generate fingerprints one at a time and record failures without stopping the loop
valid_fingerprints = {}
fingerprint_errors = {}
for row_index, molecule in molecules.loc[valid_smiles].items():
    try:
        bits = np.asarray(generator.GetFingerprintAsNumPy(molecule))
        if bits.shape != (fingerprint_size,):
            raise ValueError(f"Expected {fingerprint_size} bits, got shape {bits.shape}")
        if not np.isin(bits, (0, 1)).all():
            raise ValueError("Fingerprint contains values other than 0 and 1")
    except (RuntimeError, ValueError) as error:
        fingerprint_errors[row_index] = f"Morgan fingerprint failed: {error}"
        continue
    valid_fingerprints[row_index] = bits

# 记录失败行的原始字段和原因，供导出时核查
# Keep the original data and reason for each failed fingerprint
fingerprint_rejected = retained.loc[list(fingerprint_errors)].copy()
fingerprint_rejected["Review reason"] = pd.Series(fingerprint_errors, dtype="string")

# 使用同一个行索引，将每个分子的指纹接到它自己的记录上
# Use the same row index to append each fingerprint to its molecule's data
fingerprint_columns = [f"Morgan_{bit:04d}" for bit in range(fingerprint_size)]
fingerprints = pd.DataFrame(
    list(valid_fingerprints.values()),
    columns=fingerprint_columns,
    index=list(valid_fingerprints),
    dtype=np.uint8,
)
with_morgan = pd.concat([retained.loc[fingerprints.index], fingerprints], axis=1)

print(f"Valid fingerprints: {len(fingerprints):,}")
print(f"Morgan fingerprint failures: {len(fingerprint_rejected):,}")
if not fingerprint_rejected.empty:
    display(fingerprint_rejected[["ID", "SMILES", "Review reason"]])
display(with_morgan[["ID", *fingerprint_columns[:8]]].head())

Valid fingerprints: 9,980
Morgan fingerprint failures: 0


,ID,Morgan_0000,Morgan_0001,Morgan_0002,Morgan_0003,Morgan_0004,Morgan_0005,Morgan_0006,Morgan_0007
0,A-3,0,0,0,0,0,0,0,0
1,A-4,0,0,0,0,0,0,0,0
2,A-5,0,0,0,0,0,0,0,0
3,A-8,0,1,0,0,0,0,0,0
4,A-9,0,0,0,0,0,0,0,0


## 10. Export the Cleaned Fingerprint Dataset

The cleaned CSV contains only records that passed the structure and fingerprint checks. It keeps all original fields and appends Morgan_0000 through Morgan_1023.

The review CSV records excluded rows and their reasons, covering both unreadable SMILES and failed fingerprints.

In [11]:
# 核对记录数和唯一 ID，保证导出后可以按 ID 对齐
# Reconcile row counts and verify unique IDs for later matching
excluded = pd.concat([rejected, fingerprint_rejected]).sort_index()
assert len(df) == len(with_morgan) + len(excluded)
assert set(with_morgan.index).isdisjoint(excluded.index)
assert df["ID"].notna().all() and df["ID"].is_unique
assert np.isfinite(with_morgan[model_columns].to_numpy(dtype=float)).all()

summary = pd.Series({
    "Original rows": len(raw),
    "Name values trimmed": trim_counts["Name"],
    "SMILES values trimmed": trim_counts["SMILES"],
    "Exact duplicate rows removed": exact_duplicates,
    "Rows after original cleaning checks": len(df),
    "Records with unreadable SMILES": len(rejected),
    "Records with failed Morgan fingerprints": len(fingerprint_rejected),
    "Rows exported with Morgan fingerprints": len(with_morgan),
    "Morgan fingerprint columns": fingerprint_size,
    "Total columns in the modeling CSV": with_morgan.shape[1],
}, name="Count")
display(summary.to_frame())

# 两份 CSV 保存在同一个导出目录
# Save both CSVs in the same output directory
output_dir = Path("data") / "processed_morgan"
output_dir.mkdir(parents=True, exist_ok=True)
data_csv = output_dir / "aqsoldb_cleaned_with_morgan.csv"
review_csv = output_dir / "aqsoldb_smiles_review.csv"
with_morgan.to_csv(data_csv, index=False, encoding="utf-8-sig")
excluded.to_csv(review_csv, index=False, encoding="utf-8-sig")
print(f"Cleaned dataset: {data_csv}")
print(f"Review file: {review_csv}")

Cleaned dataset: data\processed_morgan\aqsoldb_cleaned_with_morgan.csv
Review file: data\processed_morgan\aqsoldb_smiles_review.csv


,Count
Original rows,9982
Name values trimmed,55
SMILES values trimmed,40
Exact duplicate rows removed,0
Rows after original cleaning checks,9982
Records with unreadable SMILES,2
Records with failed Morgan fingerprints,0
Rows exported with Morgan fingerprints,9980
Morgan fingerprint columns,1024
Total columns in the modeling CSV,1050
